## Replication of Ericson (2014)
Paper: Ericson, Keith. "Consumer Inertia and Firm Pricing in the Medicare Part D Prescription Drug Insurance Exchange"

Requires:
  - Data_main.dta
  - Data_subsidyinfo.dta
Produces:
  - CSV tables in Analysis_output/
  - PNG figures in Analysis_output/

Notes:
  - This is a close translation, not a line-by-line byte-identical reproduction.
  - Stata's outreg2/XML and .gph outputs are replaced with CSV and PNG outputs.
  - Some Stata factor-variable conventions are implemented with fixest formulas.
  - Cluster-robust SEs are computed at the firmID level.


### Set-up

In [3]:
# Install reuired packages
install.packages(c("haven", "tidyverse", "ggplot2", "fixest", "modelsummary"))

# Load libraries
suppressPackageStartupMessages({
  library(haven)
  library(tidyverse)    
  library(ggplot2)
  library(fixest)
  library(modelsummary)
})

# Create output directory
dir.create("Analysis_output", showWarnings = FALSE)

Installing packages into 'C:/Users/kater/AppData/Local/R/win-library/4.5'
(as 'lib' is unspecified)



package 'haven' successfully unpacked and MD5 sums checked


Warning message:
"cannot remove prior installation of package 'haven'"
Warning message in file.copy(savedcopy, lib, recursive = TRUE):
"problem copying C:\Users\kater\AppData\Local\R\win-library\4.5\00LOCK\haven\libs\x64\haven.dll to C:\Users\kater\AppData\Local\R\win-library\4.5\haven\libs\x64\haven.dll: Permission denied"
Warning message:
"restored 'haven'"


package 'tidyverse' successfully unpacked and MD5 sums checked
package 'ggplot2' successfully unpacked and MD5 sums checked
package 'fixest' successfully unpacked and MD5 sums checked


Warning message:
"cannot remove prior installation of package 'fixest'"
Warning message in file.copy(savedcopy, lib, recursive = TRUE):
"problem copying C:\Users\kater\AppData\Local\R\win-library\4.5\00LOCK\fixest\libs\x64\fixest.dll to C:\Users\kater\AppData\Local\R\win-library\4.5\fixest\libs\x64\fixest.dll: Permission denied"
Warning message:
"restored 'fixest'"


package 'modelsummary' successfully unpacked and MD5 sums checked

The downloaded binary packages are in
	C:\Users\kater\AppData\Local\Temp\RtmpAroDXb\downloaded_packages


Warning message:
"package 'tidyverse' was built under R version 4.5.3"
Warning message:
"package 'ggplot2' was built under R version 4.5.3"
Warning message:
"package 'modelsummary' was built under R version 4.5.3"


### Explore the data

In [11]:
main_data <- read_dta("../../data/Data_main.dta")
subsidies <- read_dta("../../data/Data_subsidyinfo.dta")

# Look at dimensions, variables, and first few rows of the data
dim(main_data)
names(main_data)
head(main_data)

dim(subsidies)
names(subsidies)
head(subsidies)

[1] 8382   19

[1] "orgParentCode"       "MAPlan"              "planName"           
 [4] "enrollment"          "enrollmentImpute"    "uniqueID"           
 [7] "year"                "premium"             "deductible"         
[10] "gap"                 "contractId"          "planNumber"         
[13] "btypedetail"         "state"               "PDPregion"          
[16] "LIS"                 "benefit"             "enrollmentLIS"      
[19] "enrollmentLISimpute"

orgParentCode,MAPlan,planName,enrollment,enrollmentImpute,uniqueID,year,premium,deductible,gap,contractId,planNumber,btypedetail,state,PDPregion,LIS,benefit,enrollmentLIS,enrollmentLISimpute
<chr>,<dbl>,<chr>,<dbl>,<dbl>,<chr>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<dbl>,<chr>,<chr>,<dbl>,<dbl>,<chr>,<dbl>,<dbl>
Aetna Inc.,1,Aetna Medicare Rx Essentials,494,494,S5810-037,2006,32.78,250,,S5810,37,AE,NY,3,0,B,207,207
AmeriHealth Mercy Health Plan,0,PerformRx Option II,160,160,S5650-003,2007,33.00,0,,S5650,3,BA,SC,9,1,B,48,48
"America's Health Choice Medical Plans, Inc",1,AHC Prescription Drug Plan,778,778,S9086-001,2006,48.44,250,,S9086,1,DS,FL,11,0,B,164,164
Arkansas Blue Cross Blue Shield,1,AR Blue Cross - Medi-Pak Rx Basic (PDP),32062,32062,S5795-003,2010,25.90,200,No Gap Coverage,S5795,3,BA,AR,19,1,B,NA,NA
"BCBS MN, MT, NE, ND, WY, Wellmark IA and SD",1,Blue MedicareRx - Option 2,7637,7637,S2893-002,2006,37.15,0,,S2893,2,EA,VT,2,0,E,317,317
BCBS OF AL & BCBS OF TN,1,Blue Rx Option I,6035,6035,S1030-001,2007,35.50,0,,S1030,1,BA,AL,12,0,B,210,210


[1] 34  6

[1] "PDPregion" "s2006"     "s2007"     "s2008"     "s2009"     "s2010"

PDPregion,s2006,s2007,s2008,s2009,s2010
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,36.09,30.72,30.64,28.12,26.96
2,30.27,27.35,29.17,31.74,34.57
3,29.83,24.45,24.18,27.71,33.32
4,31.37,28.12,31.23,30.99,35.01
5,33.46,29.65,30.78,30.85,33.71
6,32.59,28.45,26.59,29.23,32.09


### Create some helper functions 